# All final output

Pilot study: Deployment-Realistic Compression for Low-Resource NMT.

**What this notebook does:** [One sentence]
**Runtime:** [X hours on T4×2 / CPU-only / etc.]
**Outputs:** [Which CSV files it produces, in which folder]
**Dependencies:** See `requirements.txt` in repo root.

For context, see the project [README](../README.md) and [technical report](../writeup/technical_report.md).

In [ ]:
# Minimal install — only what Session A needs. No GPTQ dependencies.
!pip install -q transformers==4.44.2 accelerate==0.34.2 datasets==3.0.1 \
    evaluate==0.4.3 sacrebleu==2.4.3 sentencepiece==0.2.0

!pip install -q bitsandbytes==0.43.3

import torch
assert torch.cuda.is_available(), "No GPU — enable T4x2 in Kaggle settings"
print(f"torch: {torch.__version__}")
print(f"GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

In [ ]:
import os
from pathlib import Path
from datetime import datetime

BASE = Path('/kaggle/working/pilot')
for sub in ['notebooks', 'results', 'logs']:
    (BASE / sub).mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now().strftime('%Y-%m-%d_%H%M')
print(f"Run ID: {RUN_ID}")
print(f"Results folder: {BASE / 'results'}")

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

try:
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    login(token=hf_token)
    print("✓ Logged into HuggingFace")
except Exception as e:
    print(f"HF login skipped: {e}")
    print("You can still run Qwen and NLLB without a token; Aya needs it.")

In [ ]:
from datasets import load_dataset

flores_hau = load_dataset("facebook/flores", "hau_Latn", split="devtest", trust_remote_code=True)
flores_swh = load_dataset("facebook/flores", "swh_Latn", split="devtest", trust_remote_code=True)
flores_yor = load_dataset("facebook/flores", "yor_Latn", split="devtest", trust_remote_code=True)
flores_eng = load_dataset("facebook/flores", "eng_Latn", split="devtest", trust_remote_code=True)

print(f"Hausa:   {len(flores_hau)} sentences")
print(f"Swahili: {len(flores_swh)} sentences")
print(f"Yoruba:  {len(flores_yor)} sentences")
print(f"English: {len(flores_eng)} sentences")

# Sanity check — parallel sentences should align by index
print("\nSample index 0:")
print(f"  HAU: {flores_hau[0]['sentence'][:80]}")
print(f"  SWH: {flores_swh[0]['sentence'][:80]}")
print(f"  YOR: {flores_yor[0]['sentence'][:80]}")
print(f"  ENG: {flores_eng[0]['sentence'][:80]}")

In [5]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    BitsAndBytesConfig,
)
import torch
import gc

MODELS = {
    "qwen-1.5b": {
        "name": "Qwen/Qwen2.5-1.5B-Instruct",
        "arch": "causal",
        "native_quant": None,
    },
    "aya-8b": {
        "name": "CohereForAI/aya-23-8B",
        "arch": "causal",
        "native_quant": None,
    },
    "nllb-1.3b": {
        "name": "facebook/nllb-200-distilled-1.3B",
        "arch": "seq2seq",
        "native_quant": None,
    },
}

NLLB_LANG_CODES = {
    "hau": "hau_Latn",
    "swh": "swh_Latn",
    "yor": "yor_Latn",
    "eng": "eng_Latn",
}

def load_model(model_key, precision="fp16"):
    """Load model. Handles causal/seq2seq and natively-quantised checkpoints."""
    cfg = MODELS[model_key]
    model_name = cfg["name"]
    arch = cfg["arch"]
    native_quant = cfg.get("native_quant", None)

    tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side="left")
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model_cls = AutoModelForCausalLM if arch == "causal" else AutoModelForSeq2SeqLM

    if native_quant is not None:
        model = model_cls.from_pretrained(
            model_name, device_map="auto", torch_dtype=torch.float16,
        )
        label = f"native-{native_quant}"
    elif precision == "fp16":
        model = model_cls.from_pretrained(
            model_name, torch_dtype=torch.float16, device_map="auto",
        )
        label = "fp16"
    elif precision == "int8":
        bnb_cfg = BitsAndBytesConfig(load_in_8bit=True)
        model = model_cls.from_pretrained(
            model_name, quantization_config=bnb_cfg, device_map="auto",
        )
        label = "int8"
    elif precision == "int4":
        bnb_cfg = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16,
        )
        model = model_cls.from_pretrained(
            model_name, quantization_config=bnb_cfg, device_map="auto",
        )
        label = "int4"
    else:
        raise ValueError(f"Unknown precision: {precision}")

    model.eval()
    total_mem = sum(
        torch.cuda.memory_allocated(i) for i in range(torch.cuda.device_count())
    ) / 1e9
    print(f"  Loaded {model_name} ({label}), GPU memory: {total_mem:.2f} GB")
    return model, tokenizer

def free_model(model):
    """Clear GPU memory before loading next model."""
    del model
    gc.collect()
    torch.cuda.empty_cache()

In [6]:
TRANSLATION_PROMPTS = {
    ("hau", "eng"): "Translate the following Hausa sentence to English. Output only the translation, nothing else.\n\nHausa: {src}\n\nEnglish:",
    ("eng", "hau"): "Translate the following English sentence to Hausa. Output only the translation, nothing else.\n\nEnglish: {src}\n\nHausa:",
    ("swh", "eng"): "Translate the following Swahili sentence to English. Output only the translation, nothing else.\n\nSwahili: {src}\n\nEnglish:",
    ("eng", "swh"): "Translate the following English sentence to Swahili. Output only the translation, nothing else.\n\nEnglish: {src}\n\nSwahili:",
    ("yor", "eng"): "Translate the following Yoruba sentence to English. Output only the translation, nothing else.\n\nYoruba: {src}\n\nEnglish:",
    ("eng", "yor"): "Translate the following English sentence to Yoruba. Output only the translation, nothing else.\n\nEnglish: {src}\n\nYoruba:",
}

AYA_TRANSLATION_PROMPTS = {
    ("hau", "eng"): "Translate from Hausa to English: {src}",
    ("eng", "hau"): "Translate from English to Hausa: {src}",
    ("swh", "eng"): "Translate from Swahili to English: {src}",
    ("eng", "swh"): "Translate from English to Swahili: {src}",
    ("yor", "eng"): "Translate from Yoruba to English: {src}",
    ("eng", "yor"): "Translate from English to Yoruba: {src}",
}

def get_causal_prompts(model_key):
    return AYA_TRANSLATION_PROMPTS if model_key == "aya-8b" else TRANSLATION_PROMPTS


def translate_causal_batch(model, tokenizer, sentences, src_lang, tgt_lang, model_key,
                           batch_size=8, max_new_tokens=256):
    """Translate using a causal LM (Qwen family, Aya) with chat template."""
    prompts_dict = get_causal_prompts(model_key)
    prompt_tpl = prompts_dict[(src_lang, tgt_lang)]
    translations = []

    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i+batch_size]
        messages_batch = [
            [{"role": "user", "content": prompt_tpl.format(src=s)}] for s in batch
        ]
        prompts = [
            tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
            for m in messages_batch
        ]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True,
                           truncation=True, max_length=512).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                num_beams=1,
                pad_token_id=tokenizer.pad_token_id,
            )

        new_tokens = outputs[:, inputs['input_ids'].shape[1]:]
        decoded = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
        translations.extend([d.strip() for d in decoded])

        if i % (batch_size * 10) == 0:
            print(f"  Translated {i+len(batch)}/{len(sentences)}")

    return translations


def translate_nllb_batch(model, tokenizer, sentences, src_lang, tgt_lang,
                         batch_size=16, max_new_tokens=256):
    """Translate using NLLB-200 (seq2seq)."""
    src_code = NLLB_LANG_CODES[src_lang]
    tgt_code = NLLB_LANG_CODES[tgt_lang]
    tokenizer.src_lang = src_code
    forced_bos = tokenizer.convert_tokens_to_ids(tgt_code)

    translations = []
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i+batch_size]
        inputs = tokenizer(
            batch, return_tensors="pt", padding=True, truncation=True, max_length=512
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                forced_bos_token_id=forced_bos,
                max_new_tokens=max_new_tokens,
                num_beams=1,
                do_sample=False,
            )

        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        translations.extend([d.strip() for d in decoded])

        if i % (batch_size * 10) == 0:
            print(f"  Translated {i+len(batch)}/{len(sentences)}")

    return translations


def translate_batch(model, tokenizer, sentences, src_lang, tgt_lang, model_key,
                    batch_size=8, max_new_tokens=256):
    """Dispatch to the right translation function based on model architecture."""
    arch = MODELS[model_key]["arch"]
    if arch == "seq2seq":
        return translate_nllb_batch(model, tokenizer, sentences, src_lang, tgt_lang,
                                    batch_size=max(batch_size, 16),
                                    max_new_tokens=max_new_tokens)
    else:
        return translate_causal_batch(model, tokenizer, sentences, src_lang, tgt_lang,
                                      model_key, batch_size=batch_size,
                                      max_new_tokens=max_new_tokens)

In [7]:
!pip install -q sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 2.8 MB/s eta 0:00:00


In [8]:
import sacrebleu
import time
import pandas as pd

def evaluate_direction(model, tokenizer, src_data, tgt_data, src_lang, tgt_lang,
                       precision, model_key, n_samples=1012, batch_size=8):
    """Run one translation direction and compute metrics."""
    src_sents = [x['sentence'] for x in src_data.select(range(n_samples))]
    tgt_refs = [x['sentence'] for x in tgt_data.select(range(n_samples))]

    print(f"\n→ [{model_key}] {src_lang}→{tgt_lang} @ {precision}, n={n_samples}")
    t0 = time.time()
    hypotheses = translate_batch(model, tokenizer, src_sents, src_lang, tgt_lang,
                                 model_key, batch_size=batch_size)
    elapsed = time.time() - t0

    bleu = sacrebleu.corpus_bleu(hypotheses, [tgt_refs]).score
    chrf = sacrebleu.corpus_chrf(hypotheses, [tgt_refs]).score

    total_mem = sum(
        torch.cuda.max_memory_allocated(i) for i in range(torch.cuda.device_count())
    ) / 1e9

    result = {
        'run_id': RUN_ID,
        'model_key': model_key,
        'model_name': MODELS[model_key]['name'],
        'arch': MODELS[model_key]['arch'],
        'precision': precision,
        'src_lang': src_lang,
        'tgt_lang': tgt_lang,
        'direction': f"{src_lang}-{tgt_lang}",
        'n_samples': n_samples,
        'bleu': round(bleu, 2),
        'chrf': round(chrf, 2),
        'total_seconds': round(elapsed, 1),
        'sec_per_sentence': round(elapsed / n_samples, 3),
        'gpu_mem_gb': round(total_mem, 2),
    }
    print(f"  BLEU: {result['bleu']}, chrF: {result['chrf']}, time: {result['total_seconds']}s")

    hyp_path = BASE / 'results' / f'hyps_{model_key}_{src_lang}-{tgt_lang}_{precision}_{RUN_ID}.txt'
    with open(hyp_path, 'w', encoding='utf-8') as f:
        for s, h, r in zip(src_sents, hypotheses, tgt_refs):
            f.write(f"SRC: {s}\nHYP: {h}\nREF: {r}\n\n")

    return result

In [ ]:
# ============================================================
# SESSION A: Yoruba sweep across the three Week-1 models
# ============================================================

YOR_DIRECTIONS = [
    ('yor', 'eng', flores_yor, flores_eng),
    ('eng', 'yor', flores_eng, flores_yor),
]

MODEL_PRECISION_SWEEP_YOR = {
    "nllb-1.3b": ["fp16", "int8", "int4"],
    "qwen-1.5b": ["fp16", "int8", "int4"],
    "aya-8b":    ["fp16", "int8", "int4"],
}

N_SAMPLES = 1012
session_a_results = []

for model_key, precisions in MODEL_PRECISION_SWEEP_YOR.items():
    for precision in precisions:
        print(f"\n{'='*60}\n{model_key} / {precision} / YORUBA\n{'='*60}")
        torch.cuda.reset_peak_memory_stats()
        try:
            model, tokenizer = load_model(model_key, precision)
        except Exception as e:
            print(f"  ✗ Failed to load: {e}")
            continue

        for src_lang, tgt_lang, src_data, tgt_data in YOR_DIRECTIONS:
            try:
                result = evaluate_direction(
                    model, tokenizer, src_data, tgt_data,
                    src_lang, tgt_lang, precision,
                    model_key=model_key, n_samples=N_SAMPLES
                )
                session_a_results.append(result)
                pd.DataFrame(session_a_results).to_csv(
                    BASE / 'results' / f'yoruba_{RUN_ID}_partial.csv', index=False
                )
            except Exception as e:
                print(f"  ✗ Failed on {src_lang}->{tgt_lang}: {e}")

        free_model(model)

df_a = pd.DataFrame(session_a_results)
df_a.to_csv(BASE / 'results' / f'yoruba_full_{RUN_ID}.csv', index=False)
print(f"\n✓ Session A complete. Rows: {len(df_a)}")
df_a

In [ ]:
import os
from pathlib import Path
results_dir = Path('/kaggle/working/pilot/results')
for f in sorted(results_dir.glob('*.csv')):
    size = os.path.getsize(f)
    print(f"{f.name}: {size} bytes")

In [ ]:
# List all result files
import glob
result_files = sorted(glob.glob(str(BASE / 'results' / '*.csv')))
print("Result files to download:")
for f in result_files:
    print(f"  {f}")

# Also zip the hypothesis files for later error analysis
import shutil
hyp_dir = BASE / 'results'
shutil.make_archive(str(BASE / f'hyps_{RUN_ID}'), 'zip', str(hyp_dir))
print(f"\nZipped hypotheses: {BASE / f'hyps_{RUN_ID}.zip'}")

In [ ]:
# Zip only this part's outputs (the old cell zipped all of /kaggle/working into itself).
# base_dir keeps the pilot/ folder inside the zip, so it can be re-attached as a dataset to resume.
archive = shutil.make_archive(str(WORK / f"part{PART}_{MODEL_KEY}_outputs"), "zip",
                              root_dir=WORK, base_dir=OUT.name)
print(f"{archive}  ({Path(archive).stat().st_size / 1e6:.1f} MB)  | session time {hours_elapsed():.2f}h")
